# MES Workflow

This notebook explains MES execution with database-driven ERP integration and OPC-UA PLC control.

## Implemented Features and Specification Mapping

### Requirement 4.1 — MES Orders and Plant-Floor Management
- MES reads executable production lines from `mes_db.Production_Orders`.
- MES uses recipe/scheduling logic to decide operation sequence and dispatches machine actions.
- MES communicates with PLC through OPC-UA variable writes/reads via `plc_client.py`.

### Requirement 4.2 — MES User Interface
- `mes_monitor.py` provides order monitoring with:
  - current production list
  - completed order visibility
  - produced and pending quantities
  - algorithm status panel (priority ordering)
  - production time estimate per order

### Requirement 4.3 — Statistics
- Machine statistics are persisted and displayed:
  - total operating time
  - occupation indicator
  - tool changes
  - processed pieces (total/by type)
- Tool statistics:
  - operating time by tool
- Unloading statistics:
  - totals by dock
  - totals by dock and piece type

## Component Structure
- `MES/mes_server.py`: MES runtime entrypoint (database-to-control flow).
- `MES/mes_service.py`: core execution loop, status updates, and statistics persistence.
- `MES/mes_recipe_engine.py`: piece transformation and order-priority logic.
- `MES/plc_client.py`: OPC-UA node mapping and PLC command primitives.
- `MES/mes_monitor.py`: Streamlit monitoring interface.
- `MES/run_mes_programs.py`: convenience launcher.

## MES Execution Flow
1. MES polls `Production_Orders` for open production lines.
2. Recipe engine selects next action, respecting priority rules.
3. MES writes mapped OPC-UA command variables to PLC.
4. MES updates production progress and statistics tables.
5. MES monitor renders real-time state and KPI panels from database data.

### MES launcher
This code block starts the full MES program stack and opens the graphical MES monitor in your browser. The terminal only launches the service; the interface itself is a web GUI.

In [ ]:
from pathlib import Path
import subprocess, sys

root = Path.cwd()
mes_dir = root / 'MES'

def run_mes_mode(mode='all'):
    cmd = [sys.executable, str(mes_dir / 'run_mes_programs.py'), '--mode', mode]
    print('Running:', ' '.join(cmd))
    print('Modes: server, service, monitor, all')
    print('The MES monitor will open in your browser on port 8502.')
    return subprocess.Popen(cmd)

# Run the full MES component from this notebook:
# p_all = run_mes_mode('all')
# p_monitor = run_mes_mode('monitor')